# บทที่ 4 · import, object และไฟล์

บทนี้: **import · library ที่ติดตั้งด้วย pip · object / method · สร้าง object จาก class · `with` · อ่านเขียนไฟล์ · JSON**

## 4.1 import: หยิบเครื่องมือที่มีคนเขียนไว้แล้วมาใช้

Python มีเครื่องมือมาให้เยอะมาก (standard library) และติดตั้งเพิ่มได้อีกด้วย `pip`
- **module** = ไฟล์ `.py` 1 ไฟล์ · **package** = โฟลเดอร์ที่มีหลาย module

In [ ]:
import math                              # หยิบทั้งกล่อง -> ใช้ math.xxx
import random
from datetime import date                # หยิบเฉพาะบางชิ้นจากกล่อง
import pandas as pd                      # ตั้งชื่อย่อ (ธรรมเนียม: pandas = pd)

print(math.ceil(7 / 2))                  # ปัดขึ้น: แบ่งพิซซ่า 7 ชิ้นให้ 2 คน ต้องสั่งกี่ถาด
print(random.choice(["ข้าวมันไก่", "ก๋วยเตี๋ยว", "กะเพรา"]))    # สุ่มเมนูเที่ยง
print(date.today())
print(pd.__version__)

- `import math` แล้วใช้ `math.ceil(...)` : จุด `.` แปลว่า "ของที่อยู่ข้างใน"
- `from datetime import date` : หยิบ `date` ออกมาใช้ตรง ๆ
- `pandas`, `requests`, `boto3`, `duckdb` **ไม่ได้มากับ Python** ต้องติดตั้งด้วย `pip install -r requirements.txt` (ทำไปแล้วตอน setup)

⚠️ ถ้าเจอ `ModuleNotFoundError: No module named 'xxx'` = ยังไม่ได้ติดตั้ง หรือเลือก kernel ผิด (ต้องเป็น `.venv`)

In [ ]:
# import ไฟล์ .py ของเราเองก็ได้: lakehouse.py อยู่โฟลเดอร์เดียวกับ notebook นี้ (ใช้จริงในบทที่ 8)
import lakehouse

print(lakehouse.connect_lakehouse)       # เป็น function ที่เขียนไว้ในไฟล์นั้น

## 4.2 ทุกอย่างใน Python คือ object

object = **ข้อมูล + ความสามารถ** ใช้งานผ่านจุด `.`
- **attribute** (ข้อมูลของมัน) : `obj.year`
- **method** (สิ่งที่มันทำได้) : `obj.upper()`

เราใช้มาตั้งแต่บทที่ 1 แล้ว: `"latte".upper()`, `shopping_list.append("นม")`, `friend.get("email")`

## 4.3 สร้าง object จาก class

**class** = แม่พิมพ์ · **object** = ของที่ปั๊มออกมาจากแม่พิมพ์
ชื่อ class มักขึ้นต้นด้วยตัวพิมพ์ใหญ่ เรียกเหมือน function แล้วได้ object ใหม่ออกมา

**ไม่ต้องเขียน class เอง** แค่ใช้เป็น

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

birthday = datetime(2001, 5, 14, 8, 30)           # สร้าง object วันเวลา
print(birthday.year, birthday.month)              # attribute
print(birthday.strftime("%d/%m/%Y"))              # method

one_week = timedelta(days=7)                      # object "ระยะเวลา"
print(one_week.total_seconds(), "วินาที")

photos_folder = Path("../data")                   # object "ที่อยู่ไฟล์/โฟลเดอร์"
print(photos_folder.exists(), photos_folder.resolve())

💡 library ส่วนใหญ่ใช้แบบนี้: **สร้าง object ด้วยค่าตั้งต้น แล้วเรียก method ของมัน**

```python
s3 = boto3.client("s3", endpoint_url=...)    # สร้าง object สำหรับคุยกับ RustFS
s3.list_buckets()                             # เรียก method
```

## 4.4 เครื่องหมายเดียวกัน ความหมายต่างกันตามชนิดของ object

In [ ]:
print(1 + 2)                     # ตัวเลข: บวก
print("ข้าว" + "ผัด")            # ข้อความ: ต่อกัน
print(["ไข่"] + ["นม"])          # list: รวมรายการ
print(Path("data") / "marketplace" / "orders.csv")     # Path: ต่อที่อยู่ไฟล์ด้วย /

## 4.5 อ่าน/เขียนไฟล์ด้วย `with`

`with ... as ...:` = **เปิด → ใช้ → ปิดให้อัตโนมัติ** เมื่อออกจาก block (แม้จะเกิด error ระหว่างทาง)
เหมือนยืมหนังสือห้องสมุดแบบที่ระบบคืนให้เองตอนเดินออก

In [ ]:
Path("../work").mkdir(exist_ok=True)         # สร้างโฟลเดอร์ work ถ้ายังไม่มี

with open("../work/diary.txt", "w", encoding="utf-8") as f:       # "w" = เขียนใหม่
    f.write("วันนี้เรียน Python\n")
    f.write("ได้รู้จัก with แล้ว\n")

with open("../work/diary.txt", "a", encoding="utf-8") as f:       # "a" = เขียนต่อท้าย
    f.write("พรุ่งนี้เรียนต่อ\n")

with open("../work/diary.txt", encoding="utf-8") as f:            # ไม่ใส่ = อ่าน
    content = f.read()

print(content)
print("ปิดไฟล์แล้ว?", f.closed)

⚠️ ใส่ `encoding="utf-8"` ทุกครั้งที่มีภาษาไทย ไม่งั้นบน Windows อาจอ่านเป็นภาษาต่างดาว

In [ ]:
# Path ทำได้สั้นกว่าสำหรับงานง่าย ๆ และหาไฟล์ได้
note = Path("../work/note.txt")
note.write_text("ซื้อนมด้วย", encoding="utf-8")
print(note.read_text(encoding="utf-8"))

csv_files = sorted(Path("../data/marketplace").glob("*.csv"))       # หาไฟล์ที่ลงท้าย .csv
print(len(csv_files), "ไฟล์")
for p in csv_files[:3]:
    print(p.name, p.stat().st_size, "bytes")

## 4.6 JSON: เก็บ dict/list ลงไฟล์ และอ่านกลับ

JSON = รูปแบบข้อความที่หน้าตาเหมือน dict/list ของ Python ใช้ส่งข้อมูลกันทั่วโลก (เว็บ, แอป, API)

In [ ]:
import json

trip = {
    "destination": "เชียงใหม่",
    "days": 3,
    "friends": ["มานี", "ชูใจ"],
    "budget": 6500.0,
    "booked": True,
}

text = json.dumps(trip, ensure_ascii=False)       # dict -> ข้อความ JSON
print(type(text), text)

back = json.loads(text)                           # ข้อความ JSON -> dict
print(type(back), back["friends"][0])

Path("../work/trip.json").write_text(json.dumps(trip, ensure_ascii=False, indent=2), encoding="utf-8")
print(Path("../work/trip.json").read_text(encoding="utf-8"))

💡 ชนิดที่แปลงเป็น JSON ได้: dict, list, str, int, float, bool, None
(`datetime` หรือ DataFrame แปลงตรง ๆ ไม่ได้ ต้องแปลงเป็นข้อความหรือตัวเลขก่อน)

## ✅ สรุปบทที่ 4

| เรื่อง | ตัวอย่าง |
|---|---|
| import | `import math`, `from datetime import date`, `import pandas as pd`, `import lakehouse` |
| object | `.attribute`, `.method()` |
| สร้าง object | `datetime(2001, 5, 14)`, `Path("../data")`, `boto3.client("s3", ...)` |
| with | `with open(path, encoding="utf-8") as f:` |
| ไฟล์ | `Path.glob("*.csv")`, `.read_text()`, `.write_text()` |
| JSON | `json.dumps(dict)`, `json.loads(text)` |

🧪 **แบบฝึกท้ายบท:** สร้าง dict รายการของที่ต้องซื้อ (ชื่อของ + จำนวน) อย่างน้อย 3 อย่าง
บันทึกเป็นไฟล์ `../work/shopping.json` แล้วอ่านกลับมา พิมพ์ทีละบรรทัดแบบ `ไข่ x 10`

In [ ]:
# เขียนต่อตรงนี้